# Coding Session #4

---

## Today's session

Today's session is structured to teach more complex tasks in pandas and numpy. We will explore:

- More complex if statements
- Summarise by group and division by zero
- More complex indexing  

### 0. Environment preparation

We begin by loading the libraries we’ll need. In Python, libraries are like toolkits: they extend the language with specialized functions.

- **pandas (pd)** is our main tool for working with tabular data. It introduces the DataFrame, which lets us manipulate datasets in a way that feels natural if you’ve used Excel or R.
- **NumPy (np)** provides the numerical backbone. It gives us arrays and fast mathematical functions, which pandas actually uses under the hood.

In [19]:
import pandas as pd
import numpy as np


### 1. Reading Dataset

This dataset was produced by the Redistricting Data Hub (RDH) using a voter file purchased from L2, a national voter file vendor, on March 9, 2021. The RDH retrieved and processed the data on April 16, 2021.

The original L2 voter file contains individual-level voter records. RDH aggregated these records to the 2010 Census Block level, identified by a GEOID constructed by concatenating County FIPS, Census Tract, and Census Block codes. In this way they were able to ceate counts for categorical variables such as party registration, gender, and modeled commercial data (e.g., likelihood of being a homeowner or magazine subscriptions).

To have a sense of what a Census Block is, you can take a look at this pdf of the [NYC Census Tracts Map](https://www.nyc.gov/assets/planning/download/pdf/about/publications/maps/nyc-census-tracts-map.pdf)


#### Codebook

This table describes the variables included in the dataset. The daset that we use in this coding session is a subset of the original dataset and can be accessesed via [Redistricting Data Hub](https://redistrictingdatahub.org/)

| Variable | Description | Modeled |
|----------|-------------|---------|
| `party_dem` | Count of voters registered with the Democratic Party (L2 Voter File) | No |
| `party_npp` | Count of voters registered with the Non-Partisan Party (L2 Voter File) | No |
| `party_rep` | Count of voters registered with the Republican Party (L2 Voter File) | No |
| `voters_gender_m` | Count of male voters | No |
| `voters_gender_f` | Count of female voters | No |
| `voters_gender_unknkown` | Count of voters with unknown or other gender | No |
| `commercialdatall_gun_owner` | Number of voters in Census Block who own guns based on gun registrations and subscriptions to gun/hunting magazines | No |
| `commercialdatall_home_owner_or_renter_likely_homeowner` | Count of voters likely to be homeowners | Modeled |
| `commercialdatall_home_owner_or_renter_likely_renter` | Count of voters likely to be renters | Modeled |
| `commercialdata_upscalebuyerinhome_avg` | Average number of upscale buyers in the home | Modeled |
| `commercialdata_familymagazineinhome_avg` | Average number of family magazines in the home | Modeled |
| `commercialdata_femaleorientedmagazineinhome_avg` | Average number of female-oriented magazines in the home | Modeled |
| `commercialdata_financialmagazineinhome_avg` | Average number of financial magazines in the home | Modeled |
| `commercialdata_gardeningmagazineinhome_avg` | Average number of gardening magazines in the home | Modeled |
| `commercialdata_healthfitnessmagazineinhome_avg` | Average number of health and fitness magazines in the home | Modeled |



In [20]:
#Load "nyvoterfile_2021.csv" from remote repository
df_voterfile = pd.read_csv("https://raw.githubusercontent.com/albertostefanelli/data_science_campaigns/refs/heads/master/coding_sessions/session_04/data/nyvoterfile_2021.csv")

# Quick check
df_voterfile.head()

,geoid,party_dem,party_npp,party_rep,voters_gender_m,voters_gender_f,voters_gender_unknkown,commercialdatall_gun_owner,commercialdatall_home_owner_or_renter_likely_homeowner,commercialdatall_home_owner_or_renter_likely_renter,commercialdata_upscalebuyerinhome_avg,commercialdata_familymagazineinhome_avg,commercialdata_femaleorientedmagazineinhome_avg,commercialdata_financialmagazineinhome_avg,commercialdata_gardeningmagazineinhome_avg,commercialdata_healthfitnessmagazineinhome_avg
0,360593009005012,15,42,15,39,37,0,4,45,18,NaN,2.0,1.0,1.0,NaN,2.0
1,360710117023032,1,0,2,0,3,0,2,1,0,NaN,NaN,NaN,NaN,NaN,3.0
2,360550129004023,16,11,2,13,17,0,2,21,3,NaN,1.0,NaN,1.0,1.0,2.0
3,360271000001013,0,1,2,2,1,0,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN
4,360359706003096,1,0,4,3,2,0,0,2,1,NaN,NaN,NaN,NaN,1.0,2.0


### 2. More complex if statements

We want to create a new column called `majority_party` that tells us which group has the **largest share of voters** in each row: Democrats, Republicans, or Non-partisans. Importantly, we also need to account for situations where two or more groups have the **same highest share**. In those cases, the column should report “Tie”. A tie signals uncertainty and high competition, which often makes those areas strategically important. For example, in a tied precinct, even small shifts in turnout or persuasion can determine the outcome, so campaigns might invest more resources in canvassing, advertising, or get-out-the-vote drives.

You can use np.where to accomplish this. Since you want to assign values based on multiple conditions, you’ll need to nest several np.where statements. Each np.where works like an if…else check: it evaluates a condition, assigns a value if the condition is true, and otherwise falls back to another option. By nesting them, you can chain together multiple if…elif…else rules. To do this correctly, you’ll also have to combine logical operators (like &) when writing your conditions.

Let's start from the code in the last class.


In [21]:
# Previous coding session code ----------------------------------------------
# We want to calculate total voters and their share by party.

# 1. Total voters = sum of partisan categories
df_voterfile["total_voters"] = (
    df_voterfile["party_dem"]
    + df_voterfile["party_rep"]
    + df_voterfile["party_npp"]
)

# 2. Percentages by party

df_voterfile["pct_dem"] = df_voterfile["party_dem"] / df_voterfile["total_voters"]
df_voterfile["pct_rep"] = df_voterfile["party_rep"] / df_voterfile["total_voters"]
df_voterfile["pct_npp"] = df_voterfile["party_npp"] / df_voterfile["total_voters"]

# 3. Add a check column: percentages should add up to 1

df_voterfile["pct_total"] = (
    df_voterfile["pct_dem"]
    + df_voterfile["pct_rep"]
    + df_voterfile["pct_npp"]
)

# 4. Inspect the new columns
df_voterfile[["party_dem",
              "party_rep",
              "party_npp",
              "total_voters",
              "pct_dem",
              "pct_rep",
              "pct_npp",
              "pct_total"]].head()



,party_dem,party_rep,party_npp,total_voters,pct_dem,pct_rep,pct_npp,pct_total
0,15,15,42,72,0.208333,0.208333,0.583333,1.0
1,1,2,0,3,0.333333,0.666667,0.000000,1.0
2,16,2,11,29,0.551724,0.068966,0.379310,1.0
3,0,2,1,3,0.000000,0.666667,0.333333,1.0
4,1,4,0,5,0.200000,0.800000,0.000000,1.0


In [22]:
df_voterfile["majority_party"] = np.where(
    (df_voterfile["pct_dem"] > df_voterfile["pct_rep"]) &
    (df_voterfile["pct_dem"] > df_voterfile["pct_npp"]), "Dem",
    np.where(
        (df_voterfile["pct_rep"] > df_voterfile["pct_dem"]) &
        (df_voterfile["pct_rep"] > df_voterfile["pct_npp"]), "Rep",
        np.where(
            (df_voterfile["pct_npp"] > df_voterfile["pct_dem"]) &
            (df_voterfile["pct_npp"] > df_voterfile["pct_rep"]), "NPP",
            "Tie"
        )
    )
)


# Check distribution
df_voterfile["majority_party"].value_counts(normalize=True)

,proportion
majority_party,
Dem,0.454308
Rep,0.340100
NPP,0.110783
Tie,0.094809


1. **First condition:**  
   `(pct_dem > pct_rep) & (pct_dem > pct_npp)`  
   - This checks if the Democratic share is **bigger than both Republican and Non-partisan shares**.  
   - If true: label the row `"Dem"`.  

2. **Second condition:**  
   `(pct_rep > pct_dem) & (pct_rep > pct_npp)`  
   - Only runs if the first condition was false.  
   - Checks if Republicans have the largest share.  
   - If true: label the row `"Rep"`.  

3. **Third condition:**  
   `(pct_npp > pct_dem) & (pct_npp > pct_rep)`  
   - Runs if both earlier conditions were false.  
   - Checks if Non-partisans have the largest share.  
   - If true: label the row `"NPP"`.  

4. **Default (else):**  
   `"Tie"`  
   - If none of the conditions are true (e.g., two or more groups have the same max value), we assign `"Tie"`.  


### 2. Summarise by group

We have two variables in our dataset:

* **`commercialdatall_gun_owner`**: number of voters in each Census Block who own guns
* **`total_voters`**: total number of voters in each Census Block (as calculated above)

We also classified each row into **`majority_party`**: Democratic, Republican, Non-partisan (NPP), or Tie.

**We now want to**

1. **Calculate the percentage of gun owners for each Census Block**

   $$
   pct\_gunowners = \frac{commercialdatall\_gun\_owner}{total\_voters}
   $$

   This gives us the share of voters in each block who own guns.

2. **Use `groupby("majority_party")` to calculate the average percentage of gun owners across each group**
   By grouping Census Blocks by their partisan majority, we can compare how gun ownership levels differ across Democratic, Republican, Non-partisan, and Tie areas.

3. **Interpret the results**

   * Which majority group has the **highest share of gun owners**?
   * Which has the **lowest share**?


In [23]:
# Step 1: Calculate percentage gun owners per block
df_voterfile["pct_gunowners"] = df_voterfile["commercialdatall_gun_owner"] / df_voterfile["total_voters"]

In [24]:
# Step 2: Summarize by majority party
gunowner_summary = df_voterfile.groupby("majority_party")["pct_gunowners"].mean()

# Step 3: Display results
gunowner_summary

,pct_gunowners
majority_party,
Dem,0.097247
NPP,0.133039
Rep,0.175402
Tie,inf


#### Why do we have inf?

In the rows where majority_party = "Tie", at least one Census Block must have total_voters = 0. Dividing by zero gives inf (infinity). When you later take the mean by group (groupby), if one or more rows are inf, the group average can turn into inf.

Why do we have total_voters = 0? Many possibile reasons:

- The Census block/tract has no registered voters in the voter file snapshot (genuinely empty area, new development, P.O. boxes only, etc.).
- Suppressed/redacted small counts in public releases (privacy rules) get recorded as 0.
- Mismatched data (precinct→block/tract) or error in the voter file
- Voters are registered with other/minor parties not in your dataset,


In [25]:
# let's check where the total_voters are
df_voterfile[df_voterfile["total_voters"] == 0]

,geoid,party_dem,party_npp,party_rep,voters_gender_m,voters_gender_f,voters_gender_unknkown,commercialdatall_gun_owner,commercialdatall_home_owner_or_renter_likely_homeowner,commercialdatall_home_owner_or_renter_likely_renter,...,commercialdata_financialmagazineinhome_avg,commercialdata_gardeningmagazineinhome_avg,commercialdata_healthfitnessmagazineinhome_avg,total_voters,pct_dem,pct_rep,pct_npp,pct_total,majority_party,pct_gunowners
35,360290137012009,0,0,0,0,1,0,0,0,0,...,2.0,3.0,2.0,0,NaN,NaN,NaN,NaN,Tie,NaN
92,361130707011002,0,0,0,0,1,0,0,0,0,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,Tie,NaN
300,360670144002071,0,0,0,1,0,0,0,0,1,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,Tie,NaN
522,360150109001012,0,0,0,1,0,0,0,0,0,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,Tie,NaN
696,360910615001069,0,0,0,0,1,0,0,0,0,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,Tie,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237917,360870109021019,0,0,0,0,1,0,0,0,1,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,Tie,NaN
238170,360750201004046,0,0,0,2,1,0,1,0,0,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,Tie,inf
238323,361031349071039,0,0,0,1,1,0,0,2,0,...,NaN,NaN,2.0,0,NaN,NaN,NaN,NaN,Tie,NaN
238559,360690514002008,0,0,0,0,1,0,0,0,1,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,Tie,NaN


In [26]:
# let's exclude those tract where we have no registered voters for Dem, Rep, or NPP
df_voterfile_no0 = df_voterfile[df_voterfile["total_voters"] > 0]

### 3. Indexing (advanced)




Suppose we want to flag Census Blocks that are a Tie (majority_party == "Tie") and have a gun ownership rate above 0.9. We want a new column called npp_high_gunowners that equals 1 when the condition is met and 0 otherwise.
There are multiple ways to do this in pandas. In addition to np.where(), we can use

#### a. Base Pandas

- The condition produces a boolean Series: True where both conditions hold, False elsewhere.
- `astype(int)` converts True → 1 and False → 0.
- The whole column is created in one shot


In [27]:
df_voterfile["tie_high_gunowners_base"] = (
    ((df_voterfile["majority_party"] == "Tie") &
     (df_voterfile["pct_gunowners"] > 0.3))
    .astype(int)
)

# Quick check: counts of flagged rows
df_voterfile["tie_high_gunowners_base"].value_counts(normalize=True)*100

,proportion
tie_high_gunowners_base,
0,98.556277
1,1.443723


#### b. assignment with .loc indexing

- .loc is **label-based indexing**: you select rows/columns by their index labels.
- df.loc[row_labels, column_labels]

In [28]:
# Example: extract the census tract's majority_party where the index label is 1
df_voterfile.loc[1, "majority_party"]

# You can also update (replace) the value for that row:
df_voterfile.loc[1, "majority_party"] = "Independent"

Let's now see how to do a dummy assignment

In [29]:
# No initialization (NB: WRONG):

df_voterfile.loc[
    (df_voterfile["majority_party"] == "Tie") & (df_voterfile["pct_gunowners"] > 0.3),
    "tie_high_gunowners_loc"
] = 1

df_voterfile["tie_high_gunowners_loc"].value_counts(normalize=True)*100


,proportion
tie_high_gunowners_loc,
1.0,100.0


- If you don’t initialize (like in the example above), pandas only assigns 1 to matching rows. Non-matching rows stay as NaN (missing).
- If you initialize first, all rows start at 0, and matching rows become 1. You get a clean 0/1 dummy.


In [30]:
# Correct approach w initialization:
df_voterfile["tie_high_gunowners_loc"] = 0

df_voterfile.loc[
    (df_voterfile["majority_party"] == "Tie") &
    (df_voterfile["pct_gunowners"] > 0.3),
    "tie_high_gunowners_loc"
] = 1

df_voterfile["tie_high_gunowners_loc"].value_counts(normalize=True)*100



,proportion
tie_high_gunowners_loc,
0,98.556277
1,1.443723


#### c. assignment with iloc indexing

- .iloc uses **integer positions, not labels.**
- This means we cannot use the name of the column (e.g., tie_high_gunowners) for indexing as we did before
- We need first extract the integer location of the column
- To do so we can use `.get_loc("NAME_VARIABLE_TO_INDEX")`


In [31]:
df_voterfile.columns.get_loc("majority_party")

21

In [32]:
# Example: extract the majority_party in the first row (row position = 0)
df_voterfile.iloc[0, df_voterfile.columns.get_loc("majority_party")]

# You can also update (replace) the value for that row:
df_voterfile.iloc[0, df_voterfile.columns.get_loc("majority_party")] = "Independent"

- If we need to do an assignment with iloc we need to also find the row numbers (row_idx) where the condition holds.
- Then we assign 1 by integer location.

In [33]:
df_voterfile["tie_high_gunowners_iloc"] = 0

# row labels that match the condition.
row_idx = df_voterfile.index[
    (df_voterfile["majority_party"] == "Tie") &
    (df_voterfile["pct_gunowners"] > 0.3)
]

# returns the integer position of the column.
col_pos = df_voterfile.columns.get_loc("tie_high_gunowners_iloc")

df_voterfile.iloc[row_idx, col_pos] = 1

df_voterfile["tie_high_gunowners_iloc"].value_counts(normalize=True)*100


,proportion
tie_high_gunowners_iloc,
0,98.556277
1,1.443723


#### d. assignment with query indexing + loc

- .query() lets you write conditions as a string, like in SQL (WHERE).
- Returns a filtered DataFrame. We grab its index as we did before (idx).
  - The object is a label not an array so we cannout use iloc here (but see .`get_indexer(idx)`).
- We then use .loc to set 1 for those rows.
- Again, if we don’t initialize first, non-matching rows will be NaN.

In [34]:
df_voterfile["tie_high_gunowners_query"] = 0
idx = df_voterfile.query("majority_party == 'Tie' and pct_gunowners > 0.3").index
df_voterfile.loc[idx, "tie_high_gunowners_query"] = 1
df_voterfile["tie_high_gunowners_query"].value_counts(normalize=True)*100


,proportion
tie_high_gunowners_query,
0,98.556277
1,1.443723


#### Where are the non-artisan gun owners?

By creating a new flag (`npp_high_gunowners`), we can quickly identify these  cases. These rows represent areas where non-partisan voters dominate politically **and** gun ownership is extremely widespread. Let's look at geoid 360110413005007 and see how it looks like  using the [Census Interactive Map](https://tigerweb.geo.census.gov/tigerweb/)

In [35]:
df_high_gown = df_voterfile[df_voterfile["tie_high_gunowners_base"] == 1]
df_high_gown.head(40)

,geoid,party_dem,party_npp,party_rep,voters_gender_m,voters_gender_f,voters_gender_unknkown,commercialdatall_gun_owner,commercialdatall_home_owner_or_renter_likely_homeowner,commercialdatall_home_owner_or_renter_likely_renter,...,pct_dem,pct_rep,pct_npp,pct_total,majority_party,pct_gunowners,tie_high_gunowners_base,tie_high_gunowners_loc,tie_high_gunowners_iloc,tie_high_gunowners_query
11,360099605004015,0,1,1,1,2,0,1,3,0,...,0.000000,0.500000,0.500000,1.0,Tie,0.500000,1,1,1,1
93,360390811023118,5,5,3,8,6,0,4,10,0,...,0.384615,0.230769,0.384615,1.0,Tie,0.307692,1,1,1,1
129,360930331022002,4,0,4,4,5,0,3,7,0,...,0.500000,0.500000,0.000000,1.0,Tie,0.375000,1,1,1,1
235,361219703001024,0,1,1,1,2,0,2,3,0,...,0.000000,0.500000,0.500000,1.0,Tie,1.000000,1,1,1,1
236,360894906003022,0,1,1,2,1,0,1,2,0,...,0.000000,0.500000,0.500000,1.0,Tie,0.500000,1,1,1,1
353,360775903004084,1,1,0,1,1,0,1,2,0,...,0.500000,0.000000,0.500000,1.0,Tie,0.500000,1,1,1,1
367,360130369012068,4,1,4,9,3,0,3,7,0,...,0.444444,0.444444,0.111111,1.0,Tie,0.333333,1,1,1,1
422,360130368003041,3,3,2,4,4,0,4,7,0,...,0.375000,0.250000,0.375000,1.0,Tie,0.500000,1,1,1,1
435,360450605002014,0,1,1,1,1,0,2,2,0,...,0.000000,0.500000,0.500000,1.0,Tie,1.000000,1,1,1,1
447,360910615002079,1,0,1,2,1,0,2,2,0,...,0.500000,0.500000,0.000000,1.0,Tie,1.000000,1,1,1,1


# Congratulations!

You are done with the coding session. Questions or suggestions? Email Alberto at alberto.stefanelli@yale.edu

In [36]:
# Install requirements
!apt-get -qq update
!apt-get install -y pandoc texlive-xetex texlive-fonts-recommended texlive-plain-generic

from google.colab import drive, files

# Mount Google Drive
drive.mount('/content/drive')

# Ask for the notebook name
notebook_name = input(
    "Enter your notebook’s exact file name,\n"
    "exactly as shown in the top-left corner of the Colab page (next to the two yellow circle icons): "
)

# Build paths
input_path = f"/content/drive/MyDrive/Colab Notebooks/{notebook_name}"
output_path = input_path.replace(".ipynb", ".pdf")

# Convert to PDF
!jupyter nbconvert --to pdf "{input_path}"

# Download the PDF
files.download(output_path)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pandoc is already the newest version (2.9.2.1-3ubuntu2).
texlive-fonts-recommended is already the newest version (2021.20220204-1).
texlive-plain-generic is already the newest version (2021.20220204-1).
texlive-xetex is already the newest version (2021.20220204-1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Enter your notebook’s exact file name,
exactly as shown in the top-left corner of the Colab page (next to the two yellow circle icons): codingsession04.ipynb
[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/codingsession04.ipynb t

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>